# Aula 14 — Dijkstra e roteamento dinâmico

**ECAA08 · Grupo 06 · Drone agrícola de pulverização**


## Objetivo e escolha do grafo

Determinar o caminho de menor custo entre dois vértices com pesos não negativos. Aplicam-se os mesmos passos a duas redes separadas: tubulações, com custo de comprimento de mangueira; e navegação, com distância de corredores aéreos hipotéticos. Não se somam metros de tubulação a metros de voo na mesma decisão.


## Algoritmo

Inicialize $d(s)=0$ e $d(v)=\infty$ para os demais vértices. Retire da fila de prioridade o vértice de menor distância provisória. Para cada arco disponível $(u,v)$, execute a relaxação:

$$d(v)\leftarrow\min\{d(v),d(u)+w(u,v)\}.$$

Se houver melhoria, registre o predecessor de $v$ e insira o novo par na fila. Entradas antigas são ignoradas. Com pesos não negativos, retirar o destino com sua distância atual permite encerrar a busca. Sem caminho, a função retorna `(inf, [])`; não apresenta uma lista contendo apenas o destino como se fosse rota válida.

Com listas de adjacência e heap, o custo usual para esta rede simples é $O((V+E)\log V)$. Alterar disponibilidade e executar novamente é um recálculo completo; não se implementa atualização incremental.


## Exemplo hidráulico

Do tanque a `DRN_BICO_A`, a rota principal mede $0,4+0,3+0,5+0,6=1,8$ m. Bloqueando e06, a extensão pelo filtro reserva mede $0,5+0,6+0,5+0,6=2,2$ m. O ganho de disponibilidade depende de existirem fisicamente o filtro e as válvulas propostas. Não há demonstração de adequação hidráulica somente com comprimentos.


## Exemplo aéreo: BFS versus Dijkstra

| Conexão bidirecional | Distância simulada |
|---|---:|
| BASE–T1 | 300 m |
| BASE–J1 | 80 m |
| J1–T1 | 100 m |
| BASE–J2 | 120 m |
| J2–T1 | 90 m |
| T1–T2 | 70 m |
| J2–T2 | 140 m |

BFS escolhe BASE→T1, com um salto e 300 m. Dijkstra escolhe BASE→J1→T1, com dois saltos e 180 m. Interditando ambos os sentidos de J1–T1, a nova rota usa J2 e mede 210 m. As distâncias representam comprimentos de corredores, sem exigir que o arco de 300 m seja uma reta entre os pontos.

O caminho de retorno também deve existir. O exemplo verifica ida e volta e calcula tempo de deslocamento a 5 m/s; a velocidade é apenas parâmetro de simulação. Uma análise de energia ou vento exigiria outro modelo de custos. Estes trajetos são sugestões de supervisão; sua execução pertence à controladora de voo.

**Entregável:** Dijkstra com predecessor, bloqueio de nós/trechos, recálculo e verificações de custo exato, rota impossível e peso negativo rejeitado.


## Implementação executável

Python 3.10 ou superior. Execute de cima para baixo. As definições estão incluídas para permitir abrir apenas este notebook no Jupyter ou Colab. Nenhuma conexão com equipamento real é realizada.

In [1]:
from collections import deque, Counter
from math import inf, isfinite, hypot
from heapq import heappush, heappop
from itertools import permutations
from copy import deepcopy


In [2]:
def tabela(linhas):
    if not linhas:
        return '(sem registros)'
    cols = list(linhas[0])
    larg = {c: max(len(c), *(len(str(r.get(c, ''))) for r in linhas)) for c in cols}
    fmt = lambda r: ' | '.join(str(r.get(c, '')).ljust(larg[c]) for c in cols)
    return '\n'.join([fmt(dict(zip(cols, cols))), '-+-'.join('-'*larg[c] for c in cols)] + [fmt(r) for r in linhas])


In [3]:
def matriz_texto(M, linhas, colunas):
    return tabela([{'no': r, **{c: ('inf' if M[i][j] == inf else round(M[i][j], 3))
                   for j, c in enumerate(colunas)}} for i, r in enumerate(linhas)])


In [4]:
class GrafoTubulacao:
    """Dígrafo simples; catálogo físico e disponibilidade operacional separados.

    Sensores são atributos; válvulas podem estar nas arestas. Não há suposição
    de uma válvula em cada trecho. Arestas paralelas são rejeitadas.
    """
    def __init__(self, vertices):
        self.vertices = list(vertices)
        if len(set(self.vertices)) != len(self.vertices):
            raise ValueError('Vértices duplicados')
        self.arestas = {}
        self.adj = {v: [] for v in self.vertices}

    def adicionar_tubulacao(self, id_, origem, destino, comprimento_m,
                           valvula=None, diametro_mm=8.0, modos=('voo',)):
        if origem not in self.adj or destino not in self.adj:
            raise KeyError('Extremidade não cadastrada')
        if origem == destino or id_ in self.arestas:
            raise ValueError('Laço ou ID repetido')
        if any(self.arestas[e]['destino'] == destino for e in self.adj[origem]):
            raise ValueError('Aresta paralela: use um multigrafo com IDs próprios')
        if not isfinite(comprimento_m) or comprimento_m < 0 or not isfinite(diametro_mm) or diametro_mm <= 0:
            raise ValueError('Peso finito não negativo e diâmetro positivo são exigidos')
        self.arestas[id_] = dict(origem=origem, destino=destino, peso=float(comprimento_m),
                                 valvula=valvula, diametro_mm=diametro_mm,
                                 modos=tuple(modos), disponivel=True)
        self.adj[origem].append(id_)

    def vizinhos(self, u, modo=None, bloqueados=()):
        if u not in self.adj:
            raise KeyError(u)
        if u in bloqueados:
            return
        for id_ in self.adj[u]:
            a = self.arestas[id_]
            if a['disponivel'] and (modo is None or modo in a['modos']) and a['destino'] not in bloqueados:
                yield a['destino'], a['peso'], id_

    def bloquear(self, id_):
        self.arestas[id_]['disponivel'] = False

    def matrizes(self, modo=None):
        n = len(self.vertices)
        idx = {v: i for i, v in enumerate(self.vertices)}
        ids = list(self.arestas)
        A = [[0]*n for _ in range(n)]
        W = [[0.0 if i == j else inf for j in range(n)] for i in range(n)]
        B = [[0]*len(ids) for _ in range(n)]
        for j, id_ in enumerate(ids):
            a = self.arestas[id_]; u, v = idx[a['origem']], idx[a['destino']]
            # B descreve o catálogo físico, inclusive trechos indisponíveis.
            B[u][j], B[v][j] = -1, 1
            if a['disponivel'] and (modo is None or modo in a['modos']):
                A[u][v], W[u][v] = 1, a['peso']
        return A, W, B, ids

    def graus(self, modo=None):
        A, _, _, _ = self.matrizes(modo)
        return [{'no': v, 'entrada': sum(r[i] for r in A), 'saida': sum(A[i])}
                for i, v in enumerate(self.vertices)]


In [5]:
def criar_rede(redundante=False):
    nos = ['EST_TQ_01', 'EST_PMP_01', 'EST_MAN_01', 'DRN_TQ_01',
           'DRN_FLT_01', 'DRN_PMP_01', 'DRN_MAN_01'] + ['DRN_BICO_'+s for s in 'ABCD']
    if redundante:
        nos.append('DRN_FLT_02')
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao('e01', nos[0], nos[1], 1.5, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e02', nos[1], nos[2], 1.0, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e03', nos[2], nos[0], 2.0, 'EST_XV_REC', 20, ('mistura',))
    g.adicionar_tubulacao('e04', nos[2], nos[3], 3.0, 'EST_XV_01', 15, ('abastecimento',))
    g.adicionar_tubulacao('e05', nos[3], nos[4], 0.4, 'DRN_XV_F1_IN')
    g.adicionar_tubulacao('e06', nos[4], nos[5], 0.3, 'DRN_XV_F1_OUT')
    g.adicionar_tubulacao('e07', nos[5], nos[6], 0.5)
    for i, s in enumerate('ABCD'):
        g.adicionar_tubulacao(f'e{8+i:02}', nos[6], 'DRN_BICO_'+s, 0.6+0.2*i, 'DRN_XV_01'+s)
    if redundante:
        g.adicionar_tubulacao('e12', 'DRN_TQ_01', 'DRN_FLT_02', 0.5, 'DRN_XV_F2_IN')
        g.adicionar_tubulacao('e13', 'DRN_FLT_02', 'DRN_PMP_01', 0.6, 'DRN_XV_F2_OUT')
    return g


In [6]:
def reconstruir(pred, destino):
    if destino not in pred:
        return []
    caminho, u = [], destino
    while u is not None:
        caminho.append(u); u = pred[u]
    return caminho[::-1]


In [7]:
def bfs(g, origem, destino, modo=None, bloqueados=()):
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return [], {}, {}
    pred, nivel, fila = {origem: None}, {origem: 0}, deque([origem])
    while fila:
        u = fila.popleft()
        for v, _, _ in g.vizinhos(u, modo, bloqueados):
            if v not in pred:
                pred[v], nivel[v] = u, nivel[u]+1
                fila.append(v)
    return reconstruir(pred, destino), pred, nivel


In [8]:
def dfs_caminhos(g, origem, destino, modo=None, bloqueados=()):
    """Enumera caminhos simples. Custo depende do número de caminhos, não é O(V+E)."""
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return []
    resultado = []
    def visitar(u, caminho):
        if u == destino:
            resultado.append(caminho[:]); return
        for v, _, _ in g.vizinhos(u, modo, bloqueados):
            if v not in caminho:
                visitar(v, caminho+[v])
    visitar(origem, [origem])
    return resultado


In [9]:
def dijkstra(g, origem, destino, modo=None, bloqueados=()):
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return inf, []
    dist, pred, fila = {origem: 0.0}, {origem: None}, [(0.0, origem)]
    while fila:
        du, u = heappop(fila)
        if du != dist[u]:
            continue
        if u == destino:
            return du, reconstruir(pred, destino)
        for v, peso, _ in g.vizinhos(u, modo, bloqueados):
            if not isfinite(peso) or peso < 0:
                raise ValueError('Dijkstra exige pesos finitos não negativos')
            candidato = du+peso
            if candidato < dist.get(v, inf):
                dist[v], pred[v] = candidato, u
                heappush(fila, (candidato, v))
    return inf, []


In [10]:
def criar_navegacao():
    g = GrafoTubulacao(['BASE', 'J1', 'J2', 'T1', 'T2'])
    for i, (u,v,w) in enumerate([('BASE','T1',300), ('BASE','J1',80),
                                ('J1','T1',100), ('BASE','J2',120),
                                ('J2','T1',90), ('T1','T2',70), ('J2','T2',140)]):
        for sufixo, a, b in [('i',u,v), ('v',v,u)]:
            g.adicionar_tubulacao(f'n{i}{sufixo}', a,b,w,modos=('voo',))
    return g


## Experimento 1 — Custo hidráulico nominal e alternativo

In [11]:
rede=criar_rede(True)
dist,rota=dijkstra(rede,'DRN_TQ_01','DRN_BICO_A','voo')
assert abs(dist-1.8)<1e-9
print('Principal:',rota,'|',round(dist,2),'m')
rede.bloquear('e06')
dist,rota=dijkstra(rede,'DRN_TQ_01','DRN_BICO_A','voo')
assert abs(dist-2.2)<1e-9 and 'DRN_FLT_02' in rota
print('Alternativa:',rota,'|',round(dist,2),'m')


Principal: ['DRN_TQ_01', 'DRN_FLT_01', 'DRN_PMP_01', 'DRN_MAN_01', 'DRN_BICO_A'] | 1.8 m
Alternativa: ['DRN_TQ_01', 'DRN_FLT_02', 'DRN_PMP_01', 'DRN_MAN_01', 'DRN_BICO_A'] | 2.2 m


## Experimento 2 — Roteamento aéreo e interdição

In [12]:
nav=criar_navegacao()
rota_bfs=bfs(nav,'BASE','T1','voo')[0]
dist,rota=dijkstra(nav,'BASE','T1','voo')
print('BFS:',rota_bfs,'| 300 m')
print('Dijkstra:',rota,'|',dist,'m')
assert rota_bfs==['BASE','T1'] and dist==180
nav.bloquear('n2i'); nav.bloquear('n2v')
nova_dist,nova_rota=dijkstra(nav,'BASE','T1','voo')
volta,rota_volta=dijkstra(nav,'T1','BASE','voo')
assert nova_dist==210 and volta==210
print('Após interdição:',nova_rota,'|',nova_dist,'m')
print('Retorno:',rota_volta,'| tempo total a 5 m/s:',(nova_dist+volta)/5,'s')
for a in nav.arestas.values():
    if a['destino']=='T1': a['disponivel']=False
assert dijkstra(nav,'BASE','T1','voo')==(inf,[])
try:
    nav.adicionar_tubulacao('negativo','J1','J2',-1)
except ValueError: print('Peso negativo rejeitado.')
else: raise AssertionError('Peso inválido aceito')


BFS: ['BASE', 'T1'] | 300 m
Dijkstra: ['BASE', 'J1', 'T1'] | 180.0 m
Após interdição: ['BASE', 'J2', 'T1'] | 210.0 m
Retorno: ['T1', 'J2', 'BASE'] | tempo total a 5 m/s: 84.0 s
Peso negativo rejeitado.
